# 03 · Qwen translation baseline

Baseline: give Qwen a Chinese post, get English back. No slang definitions yet.

Before running (Colab):
1. Change runtime to A100.
2. Upload data/processed/usual_splits.csv (I added this to the code because I kept forgetting to add it)


## 1. Install and import

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
from google.colab import files

uploaded = files.upload() # usual_splits.csv
print(list(uploaded.keys())) # should print ['usual_splits.csv'] to make sure it uploaded the right thing

Saving rq1_matched_eval.csv to rq1_matched_eval (1).csv
['rq1_matched_eval (1).csv']


In [ ]:
# src for using a HF model (plus Assignment 3): https://medium.com/data-science/choosing-and-implementing-hugging-face-models-026d71426fbe$0
%pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 144.3 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [10]:
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [11]:
# Check that I'm using the right GPU
print(torch.cuda.get_device_name(0))

NVIDIA L4


## 2. Load the model

In [12]:
#model: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct$0
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Structure of this code is based on Assignment 3 & the model's Quickstart guide
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = model = AutoModelForCausalLM.from_pretrained( #parameters from the Quickstart documentation
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

#make sure it loaded on the right device
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded on: cuda:0


## 3. Selecting the best baseline prompt ##

Since Qwen is a chat model, we need to send it a system and user message:
- system message saying what its job is
- user message containing the post

Kat can understand Chinese, so I will design an experiment for her to select the best prompt!

I will produce a table with the original Chinese and 3 possible translations in a randomized order. Kat will select the best translation without knowing the model it came from. Then, we'll tally up the scores and select the best baseline model!

In [ ]:
prompts = {
    "prompt_1": (
        "You are a professional translator. "
        "Translate the user's Chinese social media post into natural English. "
        "Output only the English translation, with no notes or explanations."
    ),
    "prompt_2": (
        "You are a professional translator. "
        "Translate the user's Chinese social media post into English. "
        "Keep the same tone and emotion as the original. "
        "Output only the English translation, with no notes or explanations."
    ),
    "prompt_3": (
        "Translate the following Chinese text into English. "
        "Output only the translation."
    ),
}

In [ ]:
# Helper function

def translate_with_prompt(chinese_text, system_prompt):
    # build the conversation based on Quickstart guide
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chinese_text},
    ]

    # Turn into Qwen's expected text format
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # turn the text into tokens & move to the same device as the model (so it can run quickly)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # generate the answer
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

    # the output contains the prompt AND the answer, so keep only the new tokens (the answer), then turn them back into text.
    prompt_length = inputs["input_ids"].shape[1]
    answer_tokens = output_tokens[0][prompt_length:]
    english_text = tokenizer.decode(answer_tokens, skip_special_tokens=True) #only decode the answer tokens (not the prompt)

    return english_text.strip()

In [ ]:
# Get 5 slang posts and 5 non-slang posts

import random

matched_eval = pd.read_csv("rq1_matched_eval.csv")

slang_rows = matched_eval[matched_eval["role"] == "slang"]
control_rows = matched_eval[matched_eval["role"] == "control"]

# random_state makes the sample the same every time we run this
slang_sample = slang_rows.sample(5, random_state=266)
control_sample = control_rows.sample(5, random_state=266)

prompt_test_posts = pd.concat([slang_sample, control_sample])

In [ ]:
# Code the experiment!
# Build two tables:
# blind_rows: for Kat, prompt names hidden
# key_rows:answer key (which prompt is behind A, B and C on each row)

random.seed(266) # so the shuffled order is the same every run
blind_rows = []
key_rows = []

prompt_names = list(prompts.keys()) #prompt1, prompt2, prompt3
row_number = 0

for index, row in prompt_test_posts.iterrows():
    row_number = row_number + 1
    chinese_text = row["text"]

    # Translate this post with each prompt
    translations = {} # prompt name -> translation
    for prompt_name in prompt_names:
        translations[prompt_name] = translate_with_prompt(chinese_text, prompts[prompt_name])

    # Shuffle which prompt goes in column A, B and C for THIS row
    shuffled_names = prompt_names.copy()
    random.shuffle(shuffled_names)

    # The blind table row
    blind_rows.append({
        "row": row_number,
        "chinese": chinese_text,
        "translation_A": translations[shuffled_names[0]],
        "translation_B": translations[shuffled_names[1]],
        "translation_C": translations[shuffled_names[2]],
        "best (A/B/C)": "",
        "notes": "",
    })

    # The answer key row
    key_rows.append({
        "row": row_number,
        "post_id": row["post_id"],
        "role": row["role"],# slang or control
        "label": row["label"],
        "A_is": shuffled_names[0],
        "B_is": shuffled_names[1],
        "C_is": shuffled_names[2],
    })

    print("Row", row_number, "done")

blind_table = pd.DataFrame(blind_rows)
answer_key = pd.DataFrame(key_rows)

Row 1 done
Row 2 done
Row 3 done
Row 4 done
Row 5 done
Row 6 done
Row 7 done
Row 8 done
Row 9 done
Row 10 done


In [ ]:
# Save
blind_path = "/content/drive/MyDrive/ISchool/MIDS/266/prompt_comparison_blind.csv"
key_path = "/content/drive/MyDrive/ISchool/MIDS/266/prompt_comparison_answer_key.csv"

blind_table.to_csv(blind_path, index=False, encoding="utf-8-sig")
answer_key.to_csv(key_path, index=False, encoding="utf-8-sig")

## 4. Selecting the best Context-Aware prompt ##

Both the LLM and human evaluators selected Prompt 2 as the best translation in terms of including the information from the post while maintaining emotion/tone.

I'm going to do the same experiment, except this time I'm trying to figure out which way to feed the definition list is best. To make the comparison to baseline fair, I'm going to keep Prompt 2 as the main prompt, and just edit the last sentence about how to access the dictionary.

In [19]:
# Prompt 2 (the winner) is the base for every prompt
base_prompt = (
    "You are a professional translator. "
    "Translate the user's Chinese social media post into English. "
    "Keep the same tone and emotion as the original. "
    "Output only the English translation, with no notes or explanations."
)

# I want to try the slang glossary in the system message and the user message to see which performs better
# The code below writes the different "variants" for the prompts

def format_definitions(definitions_text):
    # The csv stores several definitions in one cell, separated by " || "
    # Turn that into one bullet line per term:
    lines = []
    for one_definition in str(definitions_text).split(" || "):
        lines.append("- " + one_definition.strip())
    return "\n".join(lines)


def build_messages(chinese_text, definitions_text, variant):
    # Returns the messages to send to Qwen for one post and one variant.

    glossary = format_definitions(definitions_text)

    # Glossary in the system message (no extra instruction)
    if variant == "variant_1":
        system_text = (
            base_prompt + "\n\n"
            "For reference, here are definitions of internet slang terms that may appear in this post:\n"
            + glossary
        )
        user_text= chinese_text


    # Glossary inside the user message, before the post
    elif variant == "variant_2":
        system_text = (
            base_prompt + " "
            "The message begins with a slang glossary. Use it to understand the post if slang is included, "
            "but translate only the post."
        )
        user_text= "Slang glossary:\n" + glossary + "\n\nPost:\n" + chinese_text

    # Explicit instruction to translate slang by its meaning
    elif variant == "variant_3":
        system_text = (
            base_prompt + "\n\n"
            "This post may contain Chinese internet slang. For the posts with slang, translate each slang term according to "
            "its meaning in the definitions below, not literally, and keep its emotional tone.\n"
            + glossary
        )
        user_text= chinese_text

    else:
        raise ValueError("Unknown variant: " + variant)

    return [
        {"role": "system", "content": system_text}, {"role": "user", "content": user_text},
    ]

In [20]:
# This is the same function as before essentially, but I just had to edit the input because of the way I wrote my variants
# I wrote out the whole role: xyz name: xyz in the return of the functionabove because it made the function above cleaner

def translate_messages(messages):
    # turn the conversation into Qwen's expected text format
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # turn the text into tokens & move to the same device as the model (so it can run quickly)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # generate the answer
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

    # keep only the new tokens (the answer) and turn them back into text
    prompt_length = inputs["input_ids"].shape[1]
    answer_tokens = output_tokens[0][prompt_length:]
    english_text = tokenizer.decode(answer_tokens, skip_special_tokens=True) #only decode the answer tokens (not the prompt)

    return english_text.strip()

In [21]:
# Get 5 slang posts and 5 non-slang posts

import random

matched_eval = pd.read_csv("rq1_matched_eval.csv")

slang_rows = matched_eval[matched_eval["role"] == "slang"]
control_rows = matched_eval[matched_eval["role"] == "control"]

# random_state makes the sample the same every time we run this
slang_sample = slang_rows.sample(5, random_state=2661) #different random state than last time so we get new posts
control_sample = control_rows.sample(5, random_state=2661)

dictionary_test_posts = pd.concat([slang_sample, control_sample])

# Peek at the first conversation for variant 3, to check the formatting looks right
first = dictionary_test_posts.iloc[0]
for message in build_messages(first["text"], first["slang_definitions"], "variant_3"):
    print("---", message["role"], "---")
    print(message["content"])

--- system ---
You are a professional translator. Translate the user's Chinese social media post into English. Keep the same tone and emotion as the original. Output only the English translation, with no notes or explanations.

This post may contain Chinese internet slang. For the posts with slang, translate each slang term according to its meaning in the definitions below, not literally, and keep its emotional tone.
- 打酱油: 原意指去商店购买酱油，引申为对与自己无关的事情表示不关心、不参与，或用作一种敷衍、回避的态度，体现了明哲保身或无奈的情绪。
--- user ---
每年一到国考省考各种考就特别心烦，发自内心不想考。完全不想看书就跟本不走心，根本不走心就不看书，不看书考试就打酱油，打酱油就吊脖子分数，吊脖子分数就挨骂，挨骂就生气，生气就更加不想考！死循环！资！


In [22]:
# Code the experiment! Same as above
# Build two tables:
# blind_rows: for Kat, prompt names hidden
# key_rows:answer key (which prompt is behind A, B and C on each row)

random.seed(2661) # so the shuffled order is the same every run
blind_rows = []
key_rows = []

variant_names = ["variant_1", "variant_2", "variant_3"]
row_number = 0

for index, row in dictionary_test_posts.iterrows():
    row_number = row_number + 1
    chinese_text = row["text"]

    # Translate this post with each dictionary variant
    translations = {} # variant name -> translation
    for variant in variant_names:
        messages = build_messages(chinese_text, row["slang_definitions"], variant)
        translations[variant] = translate_messages(messages)

    # Shuffle which variant goes in column A, B and C for THIS row
    shuffled_names = variant_names.copy()
    random.shuffle(shuffled_names)

    # The blind table row
    blind_rows.append({
        "row": row_number,
        "chinese": chinese_text,
        "translation_A": translations[shuffled_names[0]],
        "translation_B": translations[shuffled_names[1]],
        "translation_C": translations[shuffled_names[2]],
        "best (A/B/C)": "",
        "notes": "",
    })

    # The answer key row
    key_rows.append({
        "row": row_number,
        "post_id": row["post_id"],
        "role": row["role"],# slang or control
        "label": row["label"],
        "A_is": shuffled_names[0],
        "B_is": shuffled_names[1],
        "C_is": shuffled_names[2],
    })

    print("Row", row_number, "done")

blind_table = pd.DataFrame(blind_rows)
answer_key = pd.DataFrame(key_rows)

Row 1 done
Row 2 done
Row 3 done
Row 4 done
Row 5 done
Row 6 done
Row 7 done
Row 8 done
Row 9 done
Row 10 done


In [24]:
# Save
blind_path = "/content/drive/MyDrive/ISchool/MIDS/266/definition_comparison_blind.csv"
key_path = "/content/drive/MyDrive/ISchool/MIDS/266/definition_comparison_answer_key.csv"

# Sometimes it won't mount my Drive, so just save to the runtime folder
blind_path = "dictionary_comparison_blind.csv"
key_path = "dictionary_comparison_answer_key.csv"

blind_table.to_csv(blind_path, index=False, encoding="utf-8-sig")
answer_key.to_csv(key_path, index=False, encoding="utf-8-sig")